In [1]:
!pip install stable-baselines3==2.0.0 
!pip install numpy
!pip install tensorflow
%load_ext tensorboard
import tensorflow as tf
import pandas as pd
import numpy as np

In [2]:
import sys
import matplotlib.pyplot as plt
from IPython.display import clear_output
import requests

In [3]:
url_api = 'https://api.boptest.net'
url = "http://localhost:80"

In [4]:
sys.path.insert(0,'boptestGym')
from boptestGym.boptestGymEnv import BoptestGymEnv

In [5]:
step_period = 5 * 60

In [6]:
# Instantiate environment
env = BoptestGymEnv(url       = url,
                    testcase              = 'bestest_hydronic_heat_pump',
                    actions               = ['oveHeaPumY_u'],
                    observations          = {'reaTZon_y':(280.,310.),
                                             'weaSta_reaWeaTDryBul_y':(265.,303.),
                                             'weaSta_reaWeaHDirNor_y':(0.,862.)
                                            },
                    random_start_time     = False,
                    start_time            = 1*24*3600,
                    max_episode_length    = 365 * 24*3600,
                    warmup_period         = 24*3600,
                    predictive_period     = 0,
                    regressive_period     = None,
                    step_period           = step_period)

ConnectionError: HTTPConnectionPool(host='localhost', port=80): Max retries exceeded with url: /testcases/bestest_hydronic_heat_pump/select (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x000001C5CAE094F0>: Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it'))

In [ ]:
env?

In [ ]:
def K_to_C(x): return x - 273.15

Tsetp_low  = 21.0   # °C

In [ ]:
obs, info = env.reset()
rows = []

done = False
timesteps = 365 * 24 * step_period * 12

action = 0 

for t in range(timesteps):
    
    Tin  = obs[0]      # (K)
    Tout = obs[1]      # (K)
    Psol = obs[2]      # (W/m2)

    
    Tin_C = K_to_C(Tin)

    if Tin_C < Tsetp_low:
        action = 1
    elif Tin_C > Tsetp_low:
        action = 0

    obs_next, reward, terminated, truncated, info = env.step([action])
    done = terminated or truncated


    kpis = env.get_kpis()   

    rows.append({
        "time_seconds": t * step_period,
        "Tin_degC": K_to_C(Tin),
        "Tout_degC": K_to_C(Tout),
        "Psol_Wm2": Psol,
        "action": action,
        "energy_kWh": kpis["ener_tot"],
        "discomfort": kpis["tdis_tot"]
    })

    obs = obs_next

    if done:
        break

In [ ]:
df = pd.DataFrame(rows)

df.to_csv("local_files/rbc_5min_data.csv", index=False)

In [ ]:
env.stop()